# Preparación y estructuración de datos – Top 50 Spotify Colombia 2024

**Evidencia:** GA1-220501094-01-AA1-EV07 – Preparación, limpieza y estructuración de datos del proyecto individual con Pandas
**Aprendiz:** Jeferson Estiven Vargas · **Instructor:** Carlos Enrique Navia Torres

**Pregunta analítica:** ¿Qué géneros musicales aparecen con más frecuencia en el Top 50 diario de Spotify Colombia durante 2024?

Este notebook parte del diagnóstico de `data_quality.ipynb` y aplica las lecciones 2, 3 y 4 del material *Preparar y estructurar datos con Python*:

1. **Lección 2:** limpieza y estandarización (pipeline con `pipe()` y validación con `assert`).
2. **Lección 3:** filtrado, ordenamiento, agrupación e indicadores.
3. **Lección 4:** combinación, estructuración y exportación a `data/processed/`.

> Los archivos de `data/raw/` **no se modifican**. Todo el resultado se guarda en `data/processed/`.

## 0. Configuración

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)

# Rutas: el notebook está en notebooks/, por eso se sube un nivel con ../
RUTA_RAW = Path("../data/raw")
RUTA_PROCESSED = Path("../data/processed")
ARCHIVO_KAGGLE = RUTA_RAW / "universal_top_spotify_songs.csv"
ARCHIVO_CHARTS = RUTA_RAW / "regional-co-weekly-2024-01-11.csv"

# Alcance del proyecto
PAIS = "CO"
ANIO = 2024
TAMANO_LISTA = 50

# Resultados del diagnóstico (data_quality.ipynb) que la limpieza debe reproducir
FECHAS_MEZCLADAS_ESPERADAS = {"2024-03-14", "2024-03-16", "2024-03-18"}
DIAS_CONFIABLES_ESPERADOS = 346
FILAS_CHARTS_ESPERADAS = 200

print("Versión de pandas:", pd.__version__)

Versión de pandas: 3.1.0


---
## 1. Lección 2 – Limpieza y estandarización de datos

Orden de trabajo del material: **duplicados → tipos → nulos → texto → duplicados implícitos → verificación**.
Primero se revisa cada problema con evidencia (secciones 1.2 a 1.6) y después se empaqueta la limpieza en funciones encadenadas con `pipe()` (1.7 a 1.10).

### 1.1 Carga de los datos crudos

In [2]:
kaggle = pd.read_csv(ARCHIVO_KAGGLE, low_memory=False)   # Dataset 1: Top 50 diario de 73 listas
charts = pd.read_csv(ARCHIVO_CHARTS)                     # Dataset 2: Top 200 semanal de Colombia

print("Dataset 1 – Kaggle:", kaggle.shape)
print("Dataset 2 – Charts:", charts.shape)

Dataset 1 – Kaggle: (2110316, 25)
Dataset 2 – Charts: (200, 9)


### 1.2 Estado inicial de los datos

La función `perfil_calidad()` resume la calidad de una tabla en una fila. Se usará otra vez al final para comparar el **antes y el después**.

In [3]:
CLAVE_LISTA = ["snapshot_date", "country", "daily_rank"]   # identifica una posición en una lista diaria


def contar_textos_irregulares(df):
    """Cuenta celdas de texto con espacio invisible (\xa0), espacios dobles o espacios en los extremos."""
    total = 0
    for columna in df.select_dtypes(include=["object", "str"]).columns:
        texto = df[columna].dropna().astype(str)
        irregular = (texto.str.contains("\xa0", regex=False)
                     | texto.str.contains("  ", regex=False)
                     | (texto != texto.str.strip()))
        total += int(irregular.sum())
    return total


def perfil_calidad(df, nombre):
    """Devuelve un diccionario con los indicadores de calidad de una tabla."""
    perfil = {
        "tabla": nombre,
        "filas": len(df),
        "columnas": df.shape[1],
        "celdas_nulas": int(df.isna().sum().sum()),
        "filas_duplicadas": int(df.duplicated().sum()),
        "textos_irregulares": contar_textos_irregulares(df),
        "memoria_mb": round(df.memory_usage(deep=True).sum() / 1e6, 1),
    }
    if set(CLAVE_LISTA).issubset(df.columns):
        perfil["posiciones_repetidas"] = int(df.duplicated(subset=CLAVE_LISTA).sum())
        perfil["duracion_cero"] = int((df["duration_ms"] == 0).sum())
    if "previous_rank" in df.columns:
        perfil["previous_rank_-1"] = int((df["previous_rank"] == -1).sum())
    return perfil


def tabla_faltantes(df):
    """Nulos y porcentaje por columna, solo para las columnas que tienen nulos."""
    t = pd.DataFrame({"nulos": df.isna().sum(),
                      "porcentaje (%)": (df.isna().mean() * 100).round(2)})
    return t[t["nulos"] > 0].sort_values("nulos", ascending=False)


# Subconjunto crudo de Colombia 2024: sirve para comparar con el resultado limpio en igualdad de condiciones
kaggle_co_crudo = kaggle[(kaggle["country"] == PAIS)
                         & kaggle["snapshot_date"].str.startswith(str(ANIO))]

estado_inicial = pd.DataFrame([
    perfil_calidad(kaggle, "Kaggle crudo (todas las listas)"),
    perfil_calidad(kaggle_co_crudo, f"Kaggle crudo ({PAIS} {ANIO})"),
    perfil_calidad(charts, "Charts crudo"),
])
estado_inicial

,tabla,filas,columnas,celdas_nulas,filas_duplicadas,textos_irregulares,memoria_mb,posiciones_repetidas,duracion_cero,previous_rank_-1
0,Kaggle crudo (todas las listas),2110316,25,30448,0,268,1224.1,3686.0,29.0,NaN
1,Kaggle crudo (CO 2024),17461,25,0,0,0,10.4,33.0,0.0,NaN
2,Charts crudo,200,9,0,0,0,0.1,NaN,NaN,44.0


**Interpretación:** el dataset de Kaggle tiene 2.110.316 filas, pero solo 17.461 (0,83 %) son de Colombia 2024. Los problemas de toda la tabla (30.448 nulos, 268 textos irregulares y 29 duraciones en 0) están en otras listas: en Colombia 2024 hay 0 nulos, 0 textos irregulares y 0 duraciones en 0. Los problemas que sí afectan al alcance son las 33 posiciones repetidas y las fechas guardadas como texto. En Charts, el único problema es el faltante disfrazado: 44 valores `-1` en `previous_rank`.

### 1.3 Duplicados

Se revisan dos tipos: filas **exactamente iguales** y **posiciones repetidas** (misma fecha, país y puesto con canciones distintas), el problema más grave que encontró el diagnóstico.

In [4]:
print("Kaggle – filas exactamente duplicadas:", kaggle.duplicated().sum())
print("Charts – filas exactamente duplicadas:", charts.duplicated().sum())
print("Charts – canciones (uri) repetidas:", charts["uri"].duplicated().sum())

# Posiciones repetidas en Colombia 2024: keep=False marca todas las copias para verlas juntas
repetidas = (kaggle_co_crudo[kaggle_co_crudo.duplicated(subset=CLAVE_LISTA, keep=False)]
             .sort_values(CLAVE_LISTA))
print("\nFilas de Colombia 2024 en posiciones repetidas:", len(repetidas))
print("Fechas afectadas:", sorted(repetidas["snapshot_date"].unique()))
print("Filas por cada fecha afectada:")
print(kaggle_co_crudo[kaggle_co_crudo["snapshot_date"].isin(repetidas["snapshot_date"])]
      .groupby("snapshot_date").size())
repetidas[["snapshot_date", "daily_rank", "name", "artists"]].head(6)

Kaggle – filas exactamente duplicadas: 0
Charts – filas exactamente duplicadas: 0
Charts – canciones (uri) repetidas: 0

Filas de Colombia 2024 en posiciones repetidas: 66
Fechas afectadas: ['2024-03-14', '2024-03-16', '2024-03-18']
Filas por cada fecha afectada:
snapshot_date
2024-03-14    54
2024-03-16    55
2024-03-18    52
dtype: int64


,snapshot_date,daily_rank,name,artists
1570022,2024-03-14,2,PERRO NEGRO,"Bad Bunny, Feid"
1570023,2024-03-14,2,SI SABE FERXXO,"Blessd, Feid"
1570028,2024-03-14,8,AMARGURA,KAROL G
1570029,2024-03-14,8,LA FALDA,Myke Towers
1570030,2024-03-14,10,Feliz Cumpleaños Ferxxo,Feid
1570031,2024-03-14,10,CLASSY 101,"Feid, Young Miko"


**Interpretación:** Colombia 2024 no tiene filas exactamente duplicadas, pero sí 33 posiciones repetidas (66 filas si se cuentan las dos copias) concentradas en 3 fechas: 14, 16 y 18 de marzo. Esas listas tienen 54, 55 y 52 filas en lugar de 50, con 33 canciones de más y 22 puestos vacíos. Los 346 días restantes suman exactamente 17.300 filas, 50 por día.

**Decisión:** excluir completas las fechas con posiciones repetidas. En esas listas no se sabe qué canción ocupó realmente cada puesto, y conservarlas contaría canciones de más. Las filas exactamente duplicadas, si aparecen, se eliminan con `drop_duplicates()`.

### 1.4 Tipos de datos

In [5]:
print(kaggle[["snapshot_date", "album_release_date", "daily_rank", "popularity", "duration_ms"]].dtypes)

# Prueba de conversión sin modificar la tabla: ¿cuántas fechas no se pueden convertir?
for columna in ["snapshot_date", "album_release_date"]:
    convertida = pd.to_datetime(kaggle_co_crudo[columna], format="ISO8601", errors="coerce")
    invalidas = convertida.isna().sum() - kaggle_co_crudo[columna].isna().sum()
    print(f"{columna}: valores no convertibles = {invalidas}")

print("\nMemoria de Kaggle crudo:", round(kaggle.memory_usage(deep=True).sum() / 1e6, 1), "MB")

snapshot_date           str
album_release_date      str
daily_rank            int64
popularity            int64
duration_ms           int64
dtype: object
snapshot_date: valores no convertibles = 0
album_release_date: valores no convertibles = 0

Memoria de Kaggle crudo: 1227.5 MB


**Interpretación:** las dos fechas están guardadas como texto (`str`), pero el 100 % de los valores de Colombia 2024 se pueden convertir, así que `pd.to_datetime()` no pierde información. Las columnas numéricas usan `int64` aunque sus valores caben en tipos más pequeños.

**Decisión:** convertir `snapshot_date` y `album_release_date` a fecha con `pd.to_datetime()`. Las columnas numéricas pequeñas (posición, movimientos, popularidad) pasan a enteros de menor tamaño (`int8`, `int16`) para ahorrar memoria, y las que deberán admitir nulos usan tipos *nullable* (`Int8`, `Int64`).

### 1.5 Valores faltantes: causa antes de decidir

In [6]:
print("Kaggle – todas las listas:")
display(tabla_faltantes(kaggle))

# Hipótesis del diagnóstico: los nulos de country son la lista Global, no un error
sin_pais = kaggle[kaggle["country"].isna()]
print("Filas sin país:", len(sin_pais), "| fechas distintas:", sin_pais["snapshot_date"].nunique(),
      "| filas por fecha:", sin_pais.groupby("snapshot_date").size().unique())

print(f"\nKaggle – solo {PAIS} {ANIO}:")
display(tabla_faltantes(kaggle_co_crudo))

print("Charts – nulos reales:", int(charts.isna().sum().sum()))
print("Charts – previous_rank = -1 (faltante disfrazado):", (charts["previous_rank"] == -1).sum())
print("Charts – de ellas, con weeks_on_chart == 1:",
      ((charts["previous_rank"] == -1) & (charts["weeks_on_chart"] == 1)).sum())

Kaggle – todas las listas:


,nulos,porcentaje (%)
country,28908,1.37
album_name,822,0.04
album_release_date,659,0.03
name,30,0.00
artists,29,0.00


Filas sin país: 28908 | fechas distintas: 578 | filas por fecha: [50 52 54 53 51 47]

Kaggle – solo CO 2024:


,nulos,porcentaje (%)


Charts – nulos reales: 0
Charts – previous_rank = -1 (faltante disfrazado): 44
Charts – de ellas, con weeks_on_chart == 1: 2


| Columna | Causa | Decisión |
| --- | --- | --- |
| `country` (Kaggle) | Las 28.908 filas sin país son la lista **Global**: 578 fechas con unas 50 filas cada una | Se eliminan al filtrar `country == "CO"`: no pertenecen al alcance |
| `name`, `artists`, `album_name` (Kaggle) | Canción sin datos de catálogo. **En Colombia 2024 hay 0**: los 30, 29 y 822 nulos están en otras listas | Regla preventiva: rellenar con `"Unknown"`, porque la canción sí estuvo en la lista y no se puede borrar |
| `album_release_date` (Kaggle) | Fecha no registrada. **En Colombia 2024 hay 0**: los 659 están en otras listas | Regla preventiva: conservar el nulo, porque inventar una fecha sesgaría cualquier análisis por año |
| Características de audio con `duration_ms == 0` | Valor imposible: la medición falló | Reemplazar por nulo (`NA`) solo en esas columnas; la fila se conserva |
| `previous_rank == -1` (Charts) | La canción no estaba en la lista la semana anterior: es nueva o vuelve a entrar (solo unas pocas tienen `weeks_on_chart == 1`) | Reemplazar por nulo (`NA`): `-1` no es una posición y distorsiona los promedios |

### 1.6 Textos y categorías

In [7]:
COLUMNAS_TEXTO_KAGGLE = ["name", "artists", "album_name"]
COLUMNAS_TEXTO_CHARTS = ["artist_names", "track_name", "source"]

print("Textos irregulares – Kaggle CO 2024:", contar_textos_irregulares(kaggle_co_crudo[COLUMNAS_TEXTO_KAGGLE]))
print("Textos irregulares – Charts:", contar_textos_irregulares(charts[COLUMNAS_TEXTO_CHARTS]))

print("\nFilas con varios artistas – Kaggle CO 2024:", kaggle_co_crudo["artists"].str.contains(", ", na=False).sum(),
      "de", len(kaggle_co_crudo))
print("Filas con varios artistas – Charts:", charts["artist_names"].str.contains(", ").sum(), "de", len(charts))

print("\nIdentificadores – Charts:", charts["uri"].head(2).tolist())
print("Identificadores – Kaggle:", kaggle_co_crudo["spotify_id"].head(2).tolist())

Textos irregulares – Kaggle CO 2024: 0
Textos irregulares – Charts: 0

Filas con varios artistas – Kaggle CO 2024: 9395 de 17461
Filas con varios artistas – Charts: 100 de 200

Identificadores – Charts: ['spotify:track:7bywjHOc0wSjGGbj04XbVi', 'spotify:track:7iQXYTyuG13aoeHxGG28Nh']
Identificadores – Kaggle: ['13BDiikG6y5o5cQTK0HpW6', '3mg9jYNC1NihbthlglzrWP']


**Decisiones:**
- Quitar espacios invisibles (`\xa0`), espacios dobles y espacios en los extremos.
- **No** cambiar mayúsculas con `.str.title()`: los nombres de artistas tienen una escritura oficial (`KAROL G`, `Feid`) que se necesita para unirlos con la tabla de géneros.
- Crear `main_artist` (el primer artista de la lista) y `artist_count`, conservando la columna original. Cada canción tendrá **un solo** género, el de su artista principal, y así no se cuenta dos veces.
- Extraer el `spotify_id` de `uri` en Charts para poder cruzar los dos datasets en la Lección 4.
- Las dos variantes de la disquera *Bichota* **se conservan y se documentan**: la disquera no interviene en la pregunta analítica.

### 1.7 Funciones de limpieza reutilizables

Cada función hace **una sola cosa**, recibe un DataFrame y devuelve otro. Esa regla permite encadenarlas con `pipe()`.

In [8]:
COLUMNAS_AUDIO = ["duration_ms", "danceability", "energy", "key", "loudness", "mode",
                  "speechiness", "acousticness", "instrumentalness", "liveness",
                  "valence", "tempo", "time_signature"]


def convertir_fechas(df):
    df = df.copy()
    df["snapshot_date"] = pd.to_datetime(df["snapshot_date"], format="%Y-%m-%d")
    df["album_release_date"] = pd.to_datetime(df["album_release_date"], format="ISO8601", errors="coerce")
    return df


def filtrar_pais_anio(df, pais, anio):
    mascara = (df["country"] == pais) & (df["snapshot_date"].dt.year == anio)
    return df[mascara].reset_index(drop=True)


def excluir_listas_mezcladas(df):
    """Quita las fechas en las que una misma posición tiene más de una canción."""
    repetidas = df.duplicated(subset=CLAVE_LISTA, keep=False)
    fechas = df.loc[repetidas, "snapshot_date"].unique()
    return df[~df["snapshot_date"].isin(fechas)].reset_index(drop=True)


def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)


def optimizar_tipos_kaggle(df):
    return df.astype({
        "daily_rank": "int8", "daily_movement": "int16", "weekly_movement": "int16",
        "popularity": "int8", "duration_ms": "Int64",
        "key": "Int8", "mode": "Int8", "time_signature": "Int8",
    })


def anular_audio_duracion_cero(df):
    """Las filas con duración 0 tienen las mediciones de audio vacías: se pasan a NA."""
    df = df.copy()
    duracion_cero = df["duration_ms"] == 0
    for columna in COLUMNAS_AUDIO:
        df[columna] = df[columna].mask(duracion_cero)
    return df


def estandarizar_textos(df, columnas):
    df = df.copy()
    for columna in columnas:
        df[columna] = (df[columna]
                       .str.replace("\xa0", " ", regex=False)
                       .str.replace(r"\s+", " ", regex=True)
                       .str.strip())
    return df


def rellenar_textos_nulos(df, columnas, valor="Unknown"):
    df = df.copy()
    df[columnas] = df[columnas].fillna(valor)
    return df


def agregar_artista_principal(df, columna_artistas):
    df = df.copy()
    artistas = df[columna_artistas].str.split(", ")
    df["main_artist"] = artistas.str[0]
    df["artist_count"] = artistas.str.len().astype("int8")
    return df


def previous_rank_a_nulo(df):
    df = df.copy()
    df["previous_rank"] = df["previous_rank"].astype("Int16").mask(df["previous_rank"] == -1)
    return df


def extraer_spotify_id(df):
    df = df.copy()
    df["spotify_id"] = df["uri"].str.split(":").str[-1]
    return df

### 1.8 Pipeline de limpieza con `pipe()` e informe de cambios

`anotar()` toma una "foto" de la tabla después de cada paso y la devuelve sin cambios. Por eso se puede intercalar entre los pasos.

In [9]:
registro = []   # se reinicia en la misma celda para que una segunda ejecución no duplique el informe


def anotar(df, tabla, paso):
    registro.append({
        "tabla": tabla,
        "paso": paso,
        "filas": len(df),
        "columnas": df.shape[1],
        "nulos": int(df.isna().sum().sum()),
        "memoria_mb": round(df.memory_usage(deep=True).sum() / 1e6, 1),
    })
    return df


kaggle_limpio = (kaggle
    .pipe(anotar, "kaggle", "0 crudo")
    .pipe(convertir_fechas)
    .pipe(filtrar_pais_anio, PAIS, ANIO)
    .pipe(anotar, "kaggle", f"1 fechas + filtro {PAIS} {ANIO}")
    .pipe(excluir_listas_mezcladas)
    .pipe(anotar, "kaggle", "2 sin listas mezcladas")
    .pipe(quitar_duplicados)
    .pipe(anotar, "kaggle", "3 sin duplicados exactos")
    .pipe(optimizar_tipos_kaggle)
    .pipe(anotar, "kaggle", "4 tipos optimizados")
    .pipe(anular_audio_duracion_cero)
    .pipe(anotar, "kaggle", "5 audio con duración 0 a NA")
    .pipe(estandarizar_textos, COLUMNAS_TEXTO_KAGGLE)
    .pipe(rellenar_textos_nulos, COLUMNAS_TEXTO_KAGGLE)
    .pipe(anotar, "kaggle", "6 textos estandarizados")
    .pipe(agregar_artista_principal, "artists")
    .pipe(anotar, "kaggle", "7 artista principal"))

charts_limpio = (charts
    .pipe(anotar, "charts", "0 crudo")
    .pipe(previous_rank_a_nulo)
    .pipe(anotar, "charts", "1 previous_rank -1 a NA")
    .pipe(estandarizar_textos, COLUMNAS_TEXTO_CHARTS)
    .pipe(extraer_spotify_id)
    .pipe(agregar_artista_principal, "artist_names")
    .pipe(anotar, "charts", "2 textos, spotify_id y artista principal"))

informe_cambios = pd.DataFrame(registro)
informe_cambios

,tabla,paso,filas,columnas,nulos,memoria_mb
0,kaggle,0 crudo,2110316,25,30448,1227.5
1,kaggle,1 fechas + filtro CO 2024,17461,25,0,8.5
2,kaggle,2 sin listas mezcladas,17300,25,0,8.4
3,kaggle,3 sin duplicados exactos,17300,25,0,8.4
4,kaggle,4 tipos optimizados,17300,25,0,7.7
5,kaggle,5 audio con duración 0 a NA,17300,25,0,7.7
6,kaggle,6 textos estandarizados,17300,25,0,7.5
7,kaggle,7 artista principal,17300,27,0,8.5
8,charts,0 crudo,200,9,0,0.1
9,charts,1 previous_rank -1 a NA,200,9,44,0.1


**Interpretación del informe de cambios:** el filtro de Colombia 2024 es el paso con más impacto: reduce la tabla de 2.110.316 a 17.461 filas, elimina los 30.448 nulos (que pertenecían a otras listas) y baja la memoria de unos 1.227 MB a 8,5 MB. Excluir las 3 fechas mezcladas quita 161 filas y deja 17.300 (346 días × 50 canciones). La optimización de tipos reduce la memoria de 8,4 a 7,7 MB. Quitar duplicados exactos, anular las duraciones en 0 y estandarizar textos no cambian los datos de Colombia 2024, porque esos problemas solo existen en otras listas: se conservan como reglas de protección para versiones futuras del dataset. Al final se agregan `main_artist` y `artist_count` (27 columnas). En Charts, los 44 valores `-1` pasan a ser nulos reales y se agregan `spotify_id`, `main_artist` y `artist_count`.

### 1.9 Duplicados implícitos

Algunos duplicados solo aparecen después de estandarizar: la misma canción con nombres escritos distinto, o el mismo nombre y artista con identificadores diferentes (reediciones o versiones).

In [10]:
canciones = kaggle_limpio.drop_duplicates("spotify_id")
nombres_por_id = kaggle_limpio.groupby("spotify_id")["name"].nunique()
ids_por_cancion = canciones.groupby(["name", "main_artist"])["spotify_id"].nunique()

print("Canciones únicas (spotify_id):", len(canciones))
print("spotify_id con más de un nombre:", (nombres_por_id > 1).sum())
print("Nombre + artista con más de un spotify_id:", (ids_por_cancion > 1).sum())
ids_por_cancion[ids_por_cancion > 1].head(10)

Canciones únicas (spotify_id): 428
spotify_id con más de un nombre: 2
Nombre + artista con más de un spotify_id: 38


name                 main_artist    
ADIVINO              Myke Towers        2
APT.                 ROSÉ               2
Adonay               Rodolfo Aicardi    2
Beautiful Things     Benson Boone       2
Bellakita            Conep              2
Boquita de Caramelo  Rodolfo Aicardi    2
Cantares de Navidad  Rodolfo Aicardi    2
Cariñito             Rodolfo Aicardi    2
Chismofilia          Luis Alfonso       2
Contentoso           Luis Alfonso       2
Name: spotify_id, dtype: int64

**Interpretación:** hay 428 `spotify_id` distintos. 2 `spotify_id` tienen dos nombres (títulos corregidos) y 38 canciones (nombre + artista principal) tienen dos `spotify_id`: la misma canción publicada como sencillo y en un álbum o compilado, como *APT.* de ROSÉ.

**Decisión:** las filas **se conservan**: cada una es una aparición real en el Top 50 y ambas versiones tienen el mismo artista principal, así que no cambian el género. Para contar **canciones distintas** se usará la combinación nombre + artista principal y no el `spotify_id`, para no contar dos veces la misma canción (390 canciones y no 428).

### 1.10 Validación con `assert`

Las reglas que deben cumplir los datos limpios se convierten en una prueba. Si una falla, la ejecución se detiene y muestra cuál regla no se cumplió.

In [11]:
def validar_kaggle(df):
    filas_por_dia = df.groupby("snapshot_date").size()
    assert set(df["country"].unique()) == {PAIS}, "hay filas de otro país"
    assert (df["snapshot_date"].dt.year == ANIO).all(), "hay fechas fuera del año"
    assert not df.duplicated(subset=CLAVE_LISTA).any(), "hay posiciones repetidas"
    assert df["daily_rank"].between(1, TAMANO_LISTA).all(), "posición fuera de rango"
    assert (filas_por_dia == TAMANO_LISTA).all(), f"hay días sin {TAMANO_LISTA} canciones"
    assert len(filas_por_dia) == DIAS_CONFIABLES_ESPERADOS, f"se esperaban {DIAS_CONFIABLES_ESPERADOS} días"
    assert not (df["duration_ms"] == 0).any(), "quedan duraciones en 0"
    assert df[COLUMNAS_TEXTO_KAGGLE].notna().all().all(), "quedan textos nulos"
    assert contar_textos_irregulares(df[COLUMNAS_TEXTO_KAGGLE]) == 0, "quedan espacios irregulares"
    assert df["main_artist"].notna().all() and (df["main_artist"] != "").all(), "canciones sin artista principal"
    return "Kaggle: validación superada"


def validar_charts(df):
    assert len(df) == FILAS_CHARTS_ESPERADAS, "cambió el número de filas"
    assert not (df["previous_rank"] == -1).any(), "quedan valores -1"
    assert df["spotify_id"].str.len().eq(22).all(), "spotify_id con formato inesperado"
    assert df["spotify_id"].is_unique, "canciones repetidas"
    assert contar_textos_irregulares(df[COLUMNAS_TEXTO_CHARTS]) == 0, "quedan espacios irregulares"
    return "Charts: validación superada"


fechas_excluidas = set(kaggle_co_crudo.loc[kaggle_co_crudo.duplicated(subset=CLAVE_LISTA, keep=False),
                                           "snapshot_date"])
assert fechas_excluidas == FECHAS_MEZCLADAS_ESPERADAS, f"fechas mezcladas inesperadas: {fechas_excluidas}"

print(validar_kaggle(kaggle_limpio))
print(validar_charts(charts_limpio))

Kaggle: validación superada
Charts: validación superada


### 1.11 Calidad antes y después de la limpieza

In [12]:
estado_final = pd.DataFrame([
    perfil_calidad(kaggle_limpio, f"Kaggle limpio ({PAIS} {ANIO})"),
    perfil_calidad(charts_limpio, "Charts limpio"),
])
comparacion_calidad = pd.concat([estado_inicial, estado_final], ignore_index=True)
comparacion_calidad

,tabla,filas,columnas,celdas_nulas,filas_duplicadas,textos_irregulares,memoria_mb,posiciones_repetidas,duracion_cero,previous_rank_-1
0,Kaggle crudo (todas las listas),2110316,25,30448,0,268,1224.1,3686.0,29.0,NaN
1,Kaggle crudo (CO 2024),17461,25,0,0,0,10.4,33.0,0.0,NaN
2,Charts crudo,200,9,0,0,0,0.1,NaN,NaN,44.0
3,Kaggle limpio (CO 2024),17300,27,0,0,0,8.7,0.0,0.0,NaN
4,Charts limpio,200,12,44,0,0,0.1,NaN,NaN,0.0


**Interpretación:** después de la limpieza, Kaggle Colombia 2024 queda con 17.300 filas (346 días × 50), 0 posiciones repetidas, 0 nulos, 0 textos irregulares y 0 duraciones en 0, con 2 columnas nuevas (`main_artist` y `artist_count`). Charts conserva sus 200 filas; sus 44 nulos son los antiguos `-1`, que ahora pandas reconoce como faltantes. Ambas tablas cumplen todas las reglas de validación de la sección 1.10.

**Resultado de la Lección 2:** `kaggle_limpio` y `charts_limpio` quedan validados y listos para el filtrado y la agrupación de la Lección 3.

---
## 2. Lección 3 – Filtrado, ordenamiento y agrupación de datos

### 2.1 Concentración de artistas principales

Cada fila de `kaggle_limpio` es una aparición en el Top 50. Agrupar por `main_artist` muestra qué artistas dominan la lista y cuántos se necesitan para cubrir la mayoría de las apariciones. Esta lista es la base de la tabla de géneros.

In [13]:
artistas = (kaggle_limpio
    .groupby("main_artist")
    .agg(apariciones=("spotify_id", "size"),
         canciones=("name", "nunique"),
         dias=("snapshot_date", "nunique"),
         mejor_posicion=("daily_rank", "min"))
    .sort_values("apariciones", ascending=False)
    .reset_index())
artistas["porcentaje"] = (artistas["apariciones"] / len(kaggle_limpio) * 100).round(2)
artistas["porcentaje_acumulado"] = (artistas["apariciones"].cumsum() / len(kaggle_limpio) * 100).round(2)

print("Artistas principales distintos:", len(artistas))
print("Artistas que suman el 80 % de las apariciones:", (artistas["porcentaje_acumulado"] < 80).sum() + 1)
artistas.to_csv("../reports/main_artists.csv", index=False, encoding="utf-8-sig")
artistas.head(20)

Artistas principales distintos: 186
Artistas que suman el 80 % de las apariciones: 34


,main_artist,apariciones,canciones,dias,mejor_posicion,porcentaje,porcentaje_acumulado
0,Feid,3041,19,346,1,17.58,17.58
1,KAROL G,1554,12,346,1,8.98,26.56
2,Blessd,1160,30,344,1,6.71,33.27
3,Ryan Castro,763,15,331,1,4.41,37.68
4,Maluma,579,7,345,4,3.35,41.02
5,Luis Alfonso,450,4,222,4,2.60,43.62
6,Bad Bunny,434,8,345,2,2.51,46.13
7,Myke Towers,427,5,292,2,2.47,48.60
8,Xavi,370,3,215,2,2.14,50.74
9,Plan B,321,2,321,19,1.86,52.60


**Interpretación:** el Top 50 de Colombia 2024 tuvo 186 artistas principales, pero está muy concentrado: 34 artistas (el 18 %) suman el 80 % de las apariciones. Feid tiene el 17,58 % de todos los puestos del año (3.041 apariciones, 19 canciones, presente los 346 días), casi el doble que KAROL G (8,98 %). Los cuatro primeros (Feid, KAROL G, Blessd y Ryan Castro) son artistas urbanos colombianos y suman el 37,68 %. Por eso, el género de esos 34 artistas define la mayor parte del resultado.

### 2.2 Tabla de géneros por artista y categorización

Ningún dataset trae el género. La tabla `data/raw/artist_genres.csv` se construyó a partir de `reports/main_artists.csv`: un género por artista principal, con su nivel de confianza (`alta`, `media`, `baja` o `revisar`). Los artistas que no se pudieron identificar quedan como `Sin clasificar`.

In [14]:
GENEROS_VALIDOS = ["Urbano latino", "Regional mexicano", "Pop", "Popular colombiana", "Rap / hip hop",
                   "Salsa", "Tropical", "Vallenato", "Otros", "Sin clasificar"]

generos = pd.read_csv(RUTA_RAW / "artist_genres.csv", encoding="utf-8-sig", keep_default_na=False)

# Validaciones de la tabla de géneros antes de usarla
invalidos = set(generos["genre"]) - set(GENEROS_VALIDOS)
artistas_sin_fila = set(kaggle_limpio["main_artist"]) - set(generos["main_artist"])
assert generos["main_artist"].is_unique, "hay artistas repetidos en la tabla de géneros"
assert not invalidos, f"géneros no válidos (revisa la escritura): {invalidos}"
assert not artistas_sin_fila, f"artistas sin género asignado: {artistas_sin_fila}"

# Categorización: cada aparición recibe el género de su artista principal con map()
mapa_generos = generos.set_index("main_artist")["genre"]
mapa_confianza = generos.set_index("main_artist")["confidence"]

top50 = kaggle_limpio.assign(
    genre=kaggle_limpio["main_artist"].map(mapa_generos),
    confidence=kaggle_limpio["main_artist"].map(mapa_confianza),
    song=kaggle_limpio["name"] + " – " + kaggle_limpio["main_artist"],   # clave de canción (sección 1.9)
    month=kaggle_limpio["snapshot_date"].dt.month,
    rank_band=pd.cut(kaggle_limpio["daily_rank"], bins=[0, 10, 25, 50], labels=["Top 10", "11-25", "26-50"]),
)
assert top50["genre"].notna().all(), "quedaron apariciones sin género"

print("Artistas en la tabla de géneros:", len(generos))
print("\nApariciones según la confianza del género (%):")
print((top50["confidence"].value_counts(normalize=True) * 100).round(2))
top50[["snapshot_date", "daily_rank", "rank_band", "song", "genre"]].head()

Artistas en la tabla de géneros: 186

Apariciones según la confianza del género (%):
confidence
alta       89.73
media       7.67
revisar     2.26
baja        0.34
Name: proportion, dtype: float64


,snapshot_date,daily_rank,rank_band,song,genre
0,2024-12-31,1,Top 10,Soltera - W Sound 01 – W Sound,Urbano latino
1,2024-12-31,2,Top 10,+57 – KAROL G,Urbano latino
2,2024-12-31,3,Top 10,POLOS OPUESTOS REMIX - BLESSD VERSION – Blessd,Urbano latino
3,2024-12-31,4,Top 10,Mírame – Blessd,Urbano latino
4,2024-12-31,5,Top 10,LUNA – Feid,Urbano latino


**Interpretación:** la tabla de géneros cubre los 186 artistas y pasó las tres validaciones. El 89,73 % de las apariciones corresponde a artistas con un género de confianza alta y el 7,67 % de confianza media; solo el 2,26 % queda como `Sin clasificar` y el 0,34 % tiene confianza baja. Por eso, aunque todos los artistas sin clasificar pertenecieran a otro género, el resultado principal no cambiaría. El borrador de la tabla se elaboró con apoyo de un asistente de IA y fue revisado por el aprendiz.

### 2.3 Filtros: máscaras booleanas, `between()`, `isin()` y `query()`

In [15]:
# Máscara booleana: los diez primeros puestos de cada día
top10 = top50[top50["daily_rank"] <= 10]

# between(): la mitad baja de la lista (puestos 26 a 50, ambos incluidos)
mitad_baja = top50[top50["daily_rank"].between(26, 50)]

# isin(): géneros tradicionales colombianos y del Caribe
GENEROS_TRADICIONALES = ["Salsa", "Tropical", "Vallenato", "Popular colombiana"]
tradicionales = top50[top50["genre"].isin(GENEROS_TRADICIONALES)]

# query(): la canción número 1 de cada día
numero_uno = top50.query("daily_rank == 1")

dias = top50["snapshot_date"].nunique()
assert len(top10) == dias * 10, "el Top 10 debe tener 10 canciones por día"
assert len(numero_uno) == dias, "debe haber un número 1 por día"

print("Días analizados:", dias)
print("Apariciones en el Top 10:", len(top10))
print("Apariciones en los puestos 26-50:", len(mitad_baja))
print("Apariciones de géneros tradicionales:", len(tradicionales))
print("Días con número 1:", len(numero_uno))

# Ordenamiento: las canciones tradicionales que más días estuvieron en la lista
(tradicionales.groupby(["song", "genre"])
    .agg(dias_en_lista=("snapshot_date", "nunique"), mejor_posicion=("daily_rank", "min"))
    .sort_values(["dias_en_lista", "mejor_posicion"], ascending=[False, True])
    .head(10))

Días analizados: 346
Apariciones en el Top 10: 3460
Apariciones en los puestos 26-50: 8650
Apariciones de géneros tradicionales: 943
Días con número 1: 346


,,dias_en_lista,mejor_posicion
song,genre,,
Contentoso – Luis Alfonso,Popular colombiana,162,6
Chismofilia – Luis Alfonso,Popular colombiana,143,4
Algo Que Se Quede – Grupo Niche,Salsa,137,21
Llorarás – Dimension Latina,Salsa,130,21
Mi Palabra Contra la de Ellos - En Vivo – Luis Alfonso,Popular colombiana,127,10
Busca Un Confidente – Los Diablitos,Vallenato,21,15
MLP – Yeison Jimenez,Popular colombiana,18,13
El Perdedor – Aventura,Tropical,18,31
La Ex - En Vivo – Luis Alfonso,Popular colombiana,18,31


**Interpretación:** los filtros confirman la estructura de los datos: 346 días, 3.460 apariciones en el Top 10 (10 por día), 8.650 en los puestos 26-50 y un número 1 por día. Los géneros tradicionales (salsa, tropical, vallenato y popular colombiana) suman solo 943 apariciones (5,45 %). Sus canciones más duraderas son de Luis Alfonso (*Contentoso*, 162 días; *Chismofilia*, 143 días) y de la salsa (*Algo Que Se Quede* de Grupo Niche, 137 días; *Llorarás* de Dimensión Latina, 130 días), aunque la salsa nunca pasó del puesto 21.

### 2.4 Indicadores

| # | Indicador | Pregunta que responde |
| --- | --- | --- |
| 1 | Participación de cada género en las apariciones del Top 50 | ¿Qué géneros aparecen con más frecuencia? (pregunta analítica) |
| 2 | Canciones y artistas distintos por género | ¿El dominio viene de muchas canciones o de pocas que duran mucho? |
| 3 | Participación de cada género en el Top 10, puestos 11-25 y 26-50 | ¿El género que más aparece también domina los primeros puestos? |
| 4 | Días en el número 1 por género | ¿Qué género lidera la lista día a día? |
| 5 | Participación mensual de cada género (`pivot_table`) | ¿El dominio es estable durante el año o cambia por temporadas? |

#### Indicadores 1 y 2: participación, canciones y artistas por género

In [16]:
por_genero = (top50
    .groupby("genre")
    .agg(apariciones=("spotify_id", "size"),
         canciones=("song", "nunique"),
         artistas=("main_artist", "nunique"),
         mejor_posicion=("daily_rank", "min"),
         posicion_promedio=("daily_rank", "mean"))
    .sort_values("apariciones", ascending=False))
por_genero["porcentaje"] = (por_genero["apariciones"] / len(top50) * 100).round(2)
por_genero["apariciones_por_cancion"] = (por_genero["apariciones"] / por_genero["canciones"]).round(1)
por_genero["posicion_promedio"] = por_genero["posicion_promedio"].round(1)

assert por_genero["apariciones"].sum() == len(top50), "la suma por género debe ser el total"
assert por_genero["porcentaje"].sum().round(0) == 100
por_genero

,apariciones,canciones,artistas,mejor_posicion,posicion_promedio,porcentaje,apariciones_por_cancion
genre,,,,,,,
Urbano latino,13808,220,71,1,24.6,79.82,62.8
Regional mexicano,1046,14,10,2,25.4,6.05,74.7
Pop,787,34,24,2,34.1,4.55,23.1
Popular colombiana,494,7,2,4,28.0,2.86,70.6
Sin clasificar,391,48,40,1,31.3,2.26,8.1
Rap / hip hop,319,5,4,3,17.5,1.84,63.8
Salsa,278,7,6,17,40.0,1.61,39.7
Tropical,109,40,20,2,30.7,0.63,2.7
Vallenato,62,15,6,2,31.7,0.36,4.1


**Interpretación:** el urbano latino concentra el 79,82 % de las apariciones (13.808 de 17.300), con 220 canciones de 71 artistas: unas 4 de cada 5 posiciones del año. Le siguen el regional mexicano (6,05 %) y el pop (4,55 %). La forma de mantenerse en la lista cambia según el género: el regional mexicano y la música popular colombiana tienen pocas canciones que duran mucho (74,7 y 70,6 apariciones por canción), mientras que el pop tiene muchas canciones de corta duración (34 canciones, 23,1 apariciones cada una) y suele ubicarse más abajo (posición promedio 34,1). El tropical y el vallenato tienen muchas canciones con muy pocas apariciones (2,7 y 4,1 por canción), algo típico de las canciones de temporada.

#### Indicador 3: participación por franja de posiciones

In [17]:
franjas = top50.pivot_table(index="genre", columns="rank_band", values="spotify_id",
                            aggfunc="size", observed=True)
participacion_franjas = (franjas / franjas.sum() * 100).round(2).loc[por_genero.index]
participacion_franjas

rank_band,Top 10,11-25,26-50
genre,,,
Urbano latino,86.18,82.29,75.78
Regional mexicano,5.06,7.05,5.84
Pop,0.81,3.10,6.91
Popular colombiana,1.73,3.20,3.10
Sin clasificar,1.04,1.93,2.95
Rap / hip hop,4.54,1.62,0.90
Salsa,NaN,0.17,3.11
Tropical,0.40,0.48,0.81
Vallenato,0.20,0.15,0.54


**Interpretación:** el dominio urbano crece hacia la cima de la lista: 75,78 % de los puestos 26-50, 82,29 % de los puestos 11-25 y 86,18 % del Top 10. El pop hace lo contrario (6,91 % abajo y 0,81 % en el Top 10), y la salsa nunca entró al Top 10. El rap / hip hop es la excepción: pesa más en el Top 10 (4,54 %) que en la parte baja (0,90 %), porque sus pocas canciones llegaron a los primeros puestos.

#### Indicador 4: días en el número 1 por género

In [18]:
dias_numero_uno = (numero_uno
    .groupby("genre")
    .agg(dias_numero_uno=("snapshot_date", "nunique"), canciones_numero_uno=("song", "nunique"))
    .sort_values("dias_numero_uno", ascending=False))
dias_numero_uno["porcentaje_dias"] = (dias_numero_uno["dias_numero_uno"] / dias * 100).round(2)

print("Canciones que fueron número 1 más días:")
print(numero_uno["song"].value_counts().head(5))
dias_numero_uno

Canciones que fueron número 1 más días:
song
LUNA – Feid                       149
Mirame – Blessd                    88
Soltera - W Sound 01 – W Sound     31
+57 – KAROL G                      30
UWAIE – Kapo                       14
Name: count, dtype: int64


,dias_numero_uno,canciones_numero_uno,porcentaje_dias
genre,,,
Urbano latino,338,10,97.69
Sin clasificar,8,1,2.31


**Interpretación:** el urbano latino ocupó el número 1 en 338 de los 346 días (97,69 %), con solo 10 canciones distintas. *LUNA* de Feid fue número 1 durante 149 días, el 43 % del año. Los otros 8 días el número 1 fue una canción de un artista `Sin clasificar`; identificarlo es la primera mejora pendiente de la tabla de géneros.

#### Indicador 5: participación mensual de cada género

In [19]:
por_mes = top50.pivot_table(index="month", columns="genre", values="spotify_id",
                            aggfunc="size", fill_value=0)
participacion_mensual = (por_mes.div(por_mes.sum(axis=1), axis=0) * 100).round(1)
participacion_mensual = participacion_mensual[por_genero.index]   # columnas de mayor a menor

participacion_mensual["Tradicionales"] = participacion_mensual[
    [g for g in GENEROS_TRADICIONALES if g in participacion_mensual.columns]].sum(axis=1)
participacion_mensual

genre,Urbano latino,Regional mexicano,Pop,Popular colombiana,Sin clasificar,Rap / hip hop,Salsa,Tropical,Vallenato,Otros,Tradicionales
month,,,,,,,,,,,
1,77.9,14.8,2.6,0.1,1.9,0.0,0.3,2.1,0.3,0.1,2.8
2,81.2,13.9,0.8,1.0,1.4,1.6,0.1,0.0,0.0,0.0,1.1
3,80.6,8.4,2.5,1.5,2.9,2.7,1.2,0.3,0.0,0.0,3.0
4,82.9,9.8,3.0,0.8,2.2,0.5,0.8,0.0,0.0,0.0,1.6
5,81.0,6.2,5.0,2.6,1.4,0.0,1.3,0.0,2.5,0.0,6.4
6,75.6,4.1,3.1,7.1,3.4,1.6,3.7,0.1,1.2,0.0,12.1
7,81.2,4.0,3.7,5.5,1.7,2.6,1.1,0.1,0.1,0.0,6.8
8,81.2,2.8,6.4,4.6,0.8,4.0,0.1,0.0,0.0,0.0,4.7
9,79.7,2.0,5.3,4.8,1.5,3.1,3.3,0.0,0.0,0.3,8.1


**Interpretación:** el urbano latino se mantiene entre el 75,6 % y el 82,9 % todos los meses: su dominio es estable durante el año. Los demás géneros cambian por temporadas: el regional mexicano baja del 14,8 % en enero al 1,9 % en diciembre; el pop sube en el segundo semestre y llega al 9,5 % en noviembre; los géneros tradicionales tienen su pico en junio (12,1 %, sobre todo por la música popular colombiana y la salsa), y el tropical aparece casi solo en diciembre y enero (4,8 % y 2,1 %), la temporada de música decembrina.

### 2.5 Respuesta a la pregunta analítica

**¿Qué géneros musicales aparecen con más frecuencia en el Top 50 diario de Spotify Colombia durante 2024?**

El **urbano latino** (reguetón, trap y estilos afines) es, por mucho, el género más frecuente: ocupa el **79,82 %** de las apariciones, el **86,18 %** del Top 10 y fue número 1 en el **97,69 %** de los días. Muy lejos lo siguen el **regional mexicano (6,05 %)**, el **pop (4,55 %)** y la **música popular colombiana (2,86 %)**. Los demás géneros (rap / hip hop, salsa, tropical y vallenato) no suman el 5 %, y su presencia depende de la temporada, como la música decembrina en diciembre y enero.

**Limitaciones:** se analizaron 346 de los 366 días del año (17 sin datos y 3 excluidos por listas mezcladas); cada canción recibe un solo género, el de su artista principal; y el 2,26 % de las apariciones quedó sin clasificar.

---
## 3. Lección 4 – Combinación y estructuración de datasets

### 3.1 Relaciones entre los datasets

| Tabla | Una fila es… | Clave | Se relaciona con |
| --- | --- | --- | --- |
| `kaggle_limpio` | una aparición diaria en el Top 50 de Colombia 2024 | `snapshot_date` + `daily_rank` | `charts_limpio` por `spotify_id`; `generos` por `main_artist` |
| `charts_limpio` | una canción del Top 200 semanal (semana que termina el 2024-01-11) | `spotify_id` | `kaggle_limpio` por `spotify_id`; `generos` por `main_artist` |
| `generos` (`artist_genres.csv`) | un artista principal con su género | `main_artist` | las dos tablas anteriores por `main_artist` |

Se usa `merge()` porque las tablas comparten claves. **No se usa `concat()`**: no hay tablas con las mismas columnas que apilar, y apilar Kaggle (diario) con Charts (semanal) sería una combinación artificial que la guía pide evitar.

### 3.2 Validación cruzada con Spotify Charts

El Top 200 semanal de Spotify Charts es la fuente oficial. Se compara con las listas diarias de Kaggle de la misma semana (del viernes 2024-01-05 al jueves 2024-01-11) para comprobar que Kaggle refleja lo que publica Spotify.

In [20]:
SEMANA_INICIO, SEMANA_FIN = pd.Timestamp("2024-01-05"), pd.Timestamp("2024-01-11")

semana = top50[top50["snapshot_date"].between(SEMANA_INICIO, SEMANA_FIN)]
canciones_semana = (semana
    .groupby("spotify_id")
    .agg(name=("name", "first"), main_artist=("main_artist", "first"), genre=("genre", "first"),
         dias_en_top50=("snapshot_date", "nunique"), mejor_posicion_diaria=("daily_rank", "min"))
    .reset_index())

print("Días de esa semana con datos en Kaggle:", semana["snapshot_date"].nunique())
print("Canciones distintas en el Top 50 diario de esa semana:", len(canciones_semana))

# ¿Las canciones del Top 50 diario de Kaggle están en el Top 200 semanal oficial?
kaggle_vs_charts = canciones_semana.merge(
    charts_limpio[["spotify_id", "rank", "streams"]],
    on="spotify_id", how="left", validate="1:1", indicator=True)
print("\nKaggle → Charts:")
print(kaggle_vs_charts["_merge"].value_counts())

# ¿Las canciones del Top 50 semanal oficial aparecieron en el Top 50 diario de Kaggle?
charts_top50 = charts_limpio[charts_limpio["rank"] <= TAMANO_LISTA]
charts_vs_kaggle = charts_top50.merge(
    canciones_semana[["spotify_id", "mejor_posicion_diaria"]],
    on="spotify_id", how="left", validate="1:1", indicator=True)
print("\nCharts (Top 50) → Kaggle:")
print(charts_vs_kaggle["_merge"].value_counts())

assert len(kaggle_vs_charts) == len(canciones_semana), "el merge cambió el número de canciones"
assert len(charts_vs_kaggle) == len(charts_top50), "el merge cambió el número de canciones"

print("\nCanciones de Kaggle que no están en el Top 200 de Charts:")
kaggle_vs_charts.loc[kaggle_vs_charts["_merge"] == "left_only",
                     ["name", "main_artist", "dias_en_top50", "mejor_posicion_diaria"]]

Días de esa semana con datos en Kaggle: 7
Canciones distintas en el Top 50 diario de esa semana: 52

Kaggle → Charts:
_merge
both          49
left_only      3
right_only     0
Name: count, dtype: int64

Charts (Top 50) → Kaggle:
_merge
both          48
left_only      2
right_only     0
Name: count, dtype: int64

Canciones de Kaggle que no están en el Top 200 de Charts:


,name,main_artist,dias_en_top50,mejor_posicion_diaria
6,Solos,Tony Dize,7,34
9,PAISA,KEVIN ROLDAN,1,33
48,Es un Secreto,Plan B,7,22


**Interpretación:** _se completa con los resultados de la ejecución._

### 3.3 ¿El indicador principal se sostiene con la fuente oficial?

Se asigna el género a las canciones de Charts con un `merge` por `main_artist`. Los artistas de Charts que no aparecen en el Top 50 de Kaggle no tienen fila en la tabla de géneros y quedan como `Sin clasificar`.

In [21]:
charts_con_genero = charts_limpio.merge(generos[["main_artist", "genre"]], on="main_artist",
                                        how="left", validate="m:1", indicator=True)
assert len(charts_con_genero) == len(charts_limpio), "el merge cambió el número de filas"
print("Artistas de Charts con género asignado:")
print(charts_con_genero["_merge"].value_counts())
charts_con_genero["genre"] = charts_con_genero["genre"].fillna("Sin clasificar")

comparacion_fuentes = pd.DataFrame({
    "Kaggle: Top 50 diario de la semana (%)": semana["genre"].value_counts(normalize=True) * 100,
    "Charts: Top 50 semanal (%)": (charts_con_genero.loc[charts_con_genero["rank"] <= TAMANO_LISTA, "genre"]
                                   .value_counts(normalize=True) * 100),
}).fillna(0).round(2).sort_values("Kaggle: Top 50 diario de la semana (%)", ascending=False)
comparacion_fuentes

Artistas de Charts con género asignado:
_merge
both          157
left_only      43
right_only      0
Name: count, dtype: int64


,Kaggle: Top 50 diario de la semana (%),Charts: Top 50 semanal (%)
genre,,
Urbano latino,80.0,80.0
Regional mexicano,16.0,14.0
Sin clasificar,2.0,4.0
Pop,2.0,2.0


**Interpretación:** _se completa con los resultados de la ejecución._

### 3.4 Dataset consolidado

Tabla final del proyecto: una fila por aparición diaria en el Top 50, con su género. El género se une con `merge(validate="m:1")`: muchas apariciones pueden tener el mismo artista, pero cada artista debe tener **una sola** fila en la tabla de géneros. Si no fuera así, el `merge` fallaría en lugar de duplicar filas en silencio.

In [22]:
COLUMNAS_FINALES = (["snapshot_date", "month", "daily_rank", "rank_band", "daily_movement", "weekly_movement",
                     "spotify_id", "name", "artists", "main_artist", "artist_count", "genre", "genre_confidence",
                     "popularity", "is_explicit", "album_name", "album_release_date"] + COLUMNAS_AUDIO)

top50_consolidado = (kaggle_limpio
    .merge(generos[["main_artist", "genre", "confidence"]], on="main_artist", how="left", validate="m:1")
    .rename(columns={"confidence": "genre_confidence"})
    .assign(month=lambda df: df["snapshot_date"].dt.month,
            rank_band=lambda df: pd.cut(df["daily_rank"], bins=[0, 10, 25, 50],
                                        labels=["Top 10", "11-25", "26-50"]))
    [COLUMNAS_FINALES]
    .sort_values(["snapshot_date", "daily_rank"])
    .reset_index(drop=True))

# Integridad y consistencia
clave = ["snapshot_date", "daily_rank"]
assert len(top50_consolidado) == len(kaggle_limpio), "el merge creó o perdió filas"
assert not top50_consolidado.duplicated(subset=clave).any(), "clave fecha + posición repetida"
assert top50_consolidado["genre"].notna().all(), "apariciones sin género"
assert set(top50_consolidado["genre"]) <= set(GENEROS_VALIDOS), "géneros no válidos"
assert top50_consolidado[["snapshot_date", "daily_rank", "spotify_id", "main_artist"]].notna().all().all()
genero_map = top50.set_index(clave)["genre"].sort_index()
genero_merge = top50_consolidado.set_index(clave)["genre"].sort_index()
assert (genero_map == genero_merge).all(), "el merge no coincide con la categorización de la Lección 3"

print("Dataset consolidado:", top50_consolidado.shape)
print("Validación del dataset consolidado superada")
top50_consolidado.head()

Dataset consolidado: (17300, 30)
Validación del dataset consolidado superada


,snapshot_date,month,daily_rank,rank_band,daily_movement,weekly_movement,spotify_id,name,artists,main_artist,artist_count,genre,genre_confidence,popularity,is_explicit,album_name,album_release_date,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,2024-01-01,1,1,Top 10,0,0,7bywjHOc0wSjGGbj04XbVi,LUNA,"Feid, ATL Jacob",Feid,2,Urbano latino,alta,90,False,FERXXOCALIPSIS,2023-12-01,196800,0.774,0.860,7,-2.888,0,0.1300,0.1310,0.000000,0.1160,0.446,100.019,4
1,2024-01-01,1,2,Top 10,0,0,7iQXYTyuG13aoeHxGG28Nh,PERRO NEGRO,"Bad Bunny, Feid",Bad Bunny,2,Urbano latino,alta,94,True,nadie sabe lo que va a pasar mañana,2023-10-13,162767,0.911,0.778,5,-2.248,1,0.2620,0.0887,0.000022,0.1790,0.345,96.057,4
2,2024-01-01,1,3,Top 10,0,0,05sO6FK8E9l2cfwnEUTjHq,CUAL ES ESA,"Feid, Pirlo",Feid,2,Urbano latino,alta,79,True,FERXXOCALIPSIS,2023-12-01,172506,0.916,0.481,6,-4.483,0,0.1940,0.0179,0.000000,0.1210,0.714,95.998,4
3,2024-01-01,1,4,Top 10,0,0,5RqSsdzTNPX1uzkmlHCFvK,QLONA,"KAROL G, Peso Pluma",KAROL G,2,Urbano latino,alta,91,True,MAÑANA SERÁ BONITO (BICHOTA SEASON),2023-08-10,172797,0.841,0.738,7,-7.455,0,0.3070,0.5200,0.000000,0.0892,0.484,169.918,4
4,2024-01-01,1,5,Top 10,0,0,4bw8mcDUSRWfQo63ZTYRnU,Según Quién,"Maluma, Carin Leon",Maluma,2,Urbano latino,alta,88,True,Don Juan,2023-08-25,142334,0.909,0.762,9,-3.974,1,0.0643,0.4340,0.000000,0.0856,0.843,101.987,3


### 3.5 Tablas de resumen estructuradas

- `genre_summary`: los indicadores 1 y 2 por género.
- `genre_monthly_share`: el indicador 5 en **formato largo** (una fila por mes y género), el que prefieren `groupby` y las librerías de gráficos, obtenido con `melt()`.
- `artist_summary`: la concentración de artistas (2.1) unida a su género con `merge(validate="1:1")`.

In [23]:
resumen_generos = por_genero.reset_index()

generos_mensual = (participacion_mensual
    .drop(columns="Tradicionales")
    .reset_index()
    .melt(id_vars="month", var_name="genre", value_name="share_percentage"))

resumen_artistas = artistas.merge(generos, on="main_artist", how="left", validate="1:1")

assert resumen_generos["apariciones"].sum() == len(top50_consolidado)
assert len(generos_mensual) == participacion_mensual.shape[0] * (participacion_mensual.shape[1] - 1)
assert generos_mensual.groupby("month")["share_percentage"].sum().between(99.5, 100.5).all(), \
    "la participación mensual debe sumar 100 %"
assert len(resumen_artistas) == len(artistas) and resumen_artistas["genre"].notna().all()

print("genre_summary:", resumen_generos.shape)
print("genre_monthly_share:", generos_mensual.shape)
print("artist_summary:", resumen_artistas.shape)
generos_mensual.head(10)

genre_summary: (10, 8)
genre_monthly_share: (120, 3)
artist_summary: (186, 10)


,month,genre,share_percentage
0,1,Urbano latino,77.9
1,2,Urbano latino,81.2
2,3,Urbano latino,80.6
3,4,Urbano latino,82.9
4,5,Urbano latino,81.0
5,6,Urbano latino,75.6
6,7,Urbano latino,81.2
7,8,Urbano latino,81.2
8,9,Urbano latino,79.7
9,10,Urbano latino,78.3


### 3.6 Exportación a `data/processed/`

Se exporta en **CSV**: la guía permite CSV o Parquet, y Parquet necesitaría instalar `pyarrow`, una dependencia que no está en `requirements.txt`. Después de guardar cada archivo se vuelve a leer para comprobar que no se perdieron filas ni columnas.

In [24]:
RUTA_PROCESSED.mkdir(parents=True, exist_ok=True)

salidas = {
    "top50_co_2024_clean.csv": top50_consolidado,
    "charts_co_weekly_2024-01-11_clean.csv": charts_limpio,
    "genre_summary.csv": resumen_generos,
    "genre_monthly_share.csv": generos_mensual,
    "artist_summary.csv": resumen_artistas,
}

for nombre, df in salidas.items():
    df.to_csv(RUTA_PROCESSED / nombre, index=False, encoding="utf-8")

# Verificación de ida y vuelta: leer lo guardado y comparar tamaños
verificacion = []
for nombre, df in salidas.items():
    leido = pd.read_csv(RUTA_PROCESSED / nombre)
    assert leido.shape == df.shape, f"{nombre}: el archivo no coincide con la tabla"
    verificacion.append({"archivo": nombre, "filas": leido.shape[0], "columnas": leido.shape[1],
                         "tamano_mb": round((RUTA_PROCESSED / nombre).stat().st_size / 1e6, 2)})

print("Exportación verificada")
pd.DataFrame(verificacion)

Exportación verificada


,archivo,filas,columnas,tamano_mb
0,top50_co_2024_clean.csv,17300,30,3.80
1,charts_co_weekly_2024-01-11_clean.csv,200,12,0.03
2,genre_summary.csv,10,8,0.00
3,genre_monthly_share.csv,120,3,0.00
4,artist_summary.csv,186,10,0.01


**Interpretación:** _se completa con los resultados de la ejecución._

**Resultado de la Lección 4:** el proyecto cuenta con un dataset consolidado y validado (`top50_co_2024_clean.csv`) y con tablas de resumen listas para las siguientes etapas (visualización y análisis).